# Pharmakaksha · BP701T · Unit I · Basic Concepts of Biostatistics
**Learn · Grow · Achieve**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajreddys/learn-pharmakaksha-site/blob/main/content/BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/BP701T_Unit1_Basic_Concepts_of_Biostatistics.ipynb) &nbsp; [Open on learn.pharmakaksha.com](https://learn.pharmakaksha.com/notebooks/index.html?path=BP701T%20Biostatistics%20and%20Research%20Methodology%20%28Theory%29/BP701T_Unit1_Basic_Concepts_of_Biostatistics.ipynb)

Companion notebook for BP701T Unit I study notes, slides and video. Every worked example from the unit is here, in order, followed by lab exercises and solutions.

> **How to use this notebook:** BP701T is a hand-calculation course. Solve each example with your calculator first, using the study notes, then run the cell to check your answer. In the exam you will not have Python.

## Using learn.pharmakaksha.com
- **First run is slow:** Python loads in your browser, which takes about 10 seconds the first time.
- **Kernel indicator:** the circle at the top right is filled while a cell runs and empty when it is idle.
- **Restart:** Kernel → Restart Kernel if something gets stuck, then run the cells again from the top.
- **Saving:** your changes are saved only in this browser. Download the notebook (File → Download) to keep a copy.
- **Start over:** delete your copy in the file browser to get a fresh notebook from the site.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
print("Ready")

## Example 1 · Mean, median and mode (tablet weights, Batch A)
Notes §1.7. Hand answer: mean 250.25, median 250, mode 250 mg.

In [ ]:
A = np.array([248,252,250,247,253,251,249,250,254,246,
              250,252,248,251,249,255,250,247,251,252])
print("Sorted:", np.sort(A))
print("n =", len(A), " sum =", A.sum())
print("Mean   =", A.mean())
print("Median =", np.median(A))
print("Mode   =", stats.mode(A, keepdims=False).mode)

## Example 2 · Frequency table for hospital stay (grouped data)
Notes §1.6.

In [ ]:
stay = pd.DataFrame({"class": ["0-4","4-8","8-12","12-16","16-20","20-24"],
                     "lower": [0,4,8,12,16,20], "f": [12,18,9,6,3,2]})
stay["x"]  = stay["lower"] + 2          # mid-point, width h = 4
stay["cf"] = stay["f"].cumsum()
stay["rel_f"] = stay["f"] / stay["f"].sum()
stay["fx"] = stay["f"] * stay["x"]
stay["fx2"] = stay["f"] * stay["x"]**2
stay

## Example 3 · Grouped mean, median and mode
Notes §1.7. Hand answer: 8.08, 6.89 and 5.60 days (Mean > Median > Mode: positive skew).

In [ ]:
N, h = stay["f"].sum(), 4
mean = stay["fx"].sum() / N

def grouped_position(target):
    # returns L + (target - cf_before)/f * h for the class containing 'target'
    i = (stay["cf"] >= target).idxmax()
    cf_before = stay["cf"][i-1] if i > 0 else 0
    return stay["lower"][i] + (target - cf_before) / stay["f"][i] * h

median = grouped_position(N/2)
i = stay["f"].idxmax(); f1, f0, f2 = stay["f"][i], stay["f"][i-1], stay["f"][i+1]
mode = stay["lower"][i] + (f1 - f0) / (2*f1 - f0 - f2) * h
print(f"Mean = {mean:.2f}  Median = {median:.2f}  Mode = {mode:.2f} days")

## Example 4 · Range, variance, SD and CV (Batch A)
Notes §1.8. Hand answer: range 9, s² 5.67, s 2.38 mg, CV 0.95%.

In [ ]:
sum_x, sum_x2, n = A.sum(), (A**2).sum(), len(A)
ss = sum_x2 - sum_x**2 / n          # shortcut formula
s2 = ss / (n - 1)
print("Sum x^2 =", sum_x2, "  Sum of squares =", ss)
print(f"Range = {A.max()-A.min()}  s^2 = {s2:.2f}  s = {s2**0.5:.2f}  CV = {s2**0.5/A.mean()*100:.2f}%")
print("NumPy check, ddof=1:", round(A.std(ddof=1), 2))

**Grouped SD (hospital stay).** Hand answer: s² = 29.54, s = 5.44 days.

In [ ]:
ssg = stay["fx2"].sum() - stay["fx"].sum()**2 / N
print(f"s^2 = {ssg/(N-1):.2f}  s = {(ssg/(N-1))**0.5:.2f} days")

## Example 5 · Quartiles, skewness and kurtosis
Notes §1.9–1.10. Hand answers: A has Q1 248.25, Q3 252; stay has Q1 4.11, Q3 11.33, Sk 0.46, 0.66 and 0.23; β₂(A) = 2.41.

`method='weibull'` uses the k(n + 1)/4 position rule from the notes.

In [ ]:
q1, q3 = np.percentile(A, [25, 75], method="weibull")
print(f"Batch A: Q1 = {q1}, Q3 = {q3}, IQR = {q3-q1}")
Q1g, Q3g = grouped_position(N/4), grouped_position(3*N/4)
print(f"Stay: Q1 = {Q1g:.2f}, Q3 = {Q3g:.2f}, QD = {(Q3g-Q1g)/2:.2f}")
s = (ssg/(N-1))**0.5
print(f"Pearson Sk (mode)   = {(mean-mode)/s:.2f}")
print(f"Pearson Sk (median) = {3*(mean-median)/s:.2f}")
print(f"Bowley Sk           = {(Q3g+Q1g-2*median)/(Q3g-Q1g):.2f}")
print(f"Batch A beta2 = {stats.kurtosis(A, fisher=False):.2f} (3 = normal)")

## Example 6 · Worked example: Batch A vs Batch B and the IP weight test
Notes, worked pharma example.

In [ ]:
B = np.array([238,262,245,256,250,241,259,247,253,244,
              257,251,236,264,249,252,243,258,246,255])
def summary(x):
    q1, q3 = np.percentile(x, [25, 75], method="weibull")
    return {"mean": x.mean(), "median": np.median(x), "SD": x.std(ddof=1),
            "CV %": x.std(ddof=1)/x.mean()*100, "range": x.max()-x.min(),
            "Q1": q1, "Q3": q3, "IQR": q3-q1}
print(pd.DataFrame({"Batch A": summary(A), "Batch B": summary(B)}).round(2))

def ip_weight_test(x):
    m = x.mean()
    out5  = x[(x < 0.95*m) | (x > 1.05*m)]
    out10 = x[(x < 0.90*m) | (x > 1.10*m)]
    verdict = "PASS" if len(out5) <= 2 and len(out10) == 0 else "FAIL"
    return f"outside 5%: {list(out5)}, outside 10%: {list(out10)} -> {verdict}"
print("A:", ip_weight_test(A)); print("B:", ip_weight_test(B))

## Example 7 · Diagrams: histogram, ogive and box plot
Notes §1.11. Read the median off the ogive at cf = 25.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].bar(stay["lower"], stay["f"], width=4, align="edge", edgecolor="black", color="#1F8A70")
ax[0].set(title="Histogram: hospital stay", xlabel="Days", ylabel="Patients")
upper = stay["lower"] + 4
ax[1].plot([0]+list(upper), [0]+list(stay["cf"]), marker="o", color="#E07A2E")
ax[1].axhline(25, ls="--", color="grey"); ax[1].axvline(median, ls="--", color="grey")
ax[1].set(title=f"Less-than ogive (median = {median:.2f})", xlabel="Days", ylabel="Cumulative frequency")
ax[2].boxplot([A, B]); ax[2].set_xticks([1, 2], ["Batch A", "Batch B"])
ax[2].set(title="Tablet weights", ylabel="mg")
plt.tight_layout(); plt.show()

## Lab exercises
1. N1 from the notes: weekly ADR reports 12, 15, 11, 18, 14, 15, 20, 13. Find mean, median, mode, range, SD and CV.
2. N2: ages of 40 hypertension patients (classes 20–30 … 60–70, f = 4, 9, 13, 10, 4). Find grouped mean, median, mode and SD.
3. A pie chart of dosage forms: tablets 45, capsules 25, syrups 15, injections 15. Find each angle and draw it.

Solve by hand first, then check below.

### Solutions

In [ ]:
x = np.array([12,15,11,18,14,15,20,13])
print("1)", x.mean(), np.median(x), stats.mode(x, keepdims=False).mode, x.max()-x.min(),
      round(x.std(ddof=1), 2), round(x.std(ddof=1)/x.mean()*100, 1), "%")
f = np.array([4,9,13,10,4]); mid = np.array([25,35,45,55,65]); N2 = f.sum()
m = (f*mid).sum()/N2; cf = f.cumsum()
med = 40 + (N2/2 - cf[1]) / f[2] * 10
mo  = 40 + (f[2]-f[1]) / (2*f[2]-f[1]-f[3]) * 10
sd  = (((f*mid**2).sum() - N2*m**2) / (N2-1))**0.5
print(f"2) mean {m:.2f}, median {med:.2f}, mode {mo:.2f}, SD {sd:.2f}")
forms = pd.Series({"Tablets":45, "Capsules":25, "Syrups":15, "Injections":15})
print("3) angles:", dict(forms/forms.sum()*360))
forms.plot.pie(autopct="%1.0f%%", figsize=(4,4)); plt.ylabel(""); plt.show()